# 04 — Inferencia y demo

Pruebas de inferencia interactiva y comparación base vs fine-tuned.

In [ ]:
!pip install -q transformers accelerate torch

In [ ]:
import torch
from transformers import AutoTokenizer, pipeline

FINETUNED_ID = 'lopezinsua/mistral-7b-code-reviewer-es'
BASE_ID = 'mistralai/Mistral-7B-Instruct-v0.3'

tokenizer = AutoTokenizer.from_pretrained(FINETUNED_ID)

pipe_ft = pipeline(
    'text-generation',
    model=FINETUNED_ID,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map='auto',
)
print('Fine-tuned model cargado')

## Función de inferencia

In [ ]:
def review(code: str, pipe, max_new_tokens: int = 300) -> str:
    prompt = f"<s>[INST] Revisa este código Python y da feedback detallado:\n\n{code} [/INST]"
    result = pipe(prompt, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    generated = result[0]['generated_text']
    return generated.split('[/INST]')[-1].strip().replace('</s>', '').strip()

## Pruebas interactivas

In [ ]:
test_cases = [
    # (descripción, código)
    ('Error lógico', 'def suma(a, b):\n  return a - b'),
    ('Indentación', 'for i in range(10):\nprint(i)'),
    ('División por cero', 'def div(a, b):\n    return a / b\n\nprint(div(5, 0))'),
    ('IndexError', 'nums = [1, 2, 3]\nprint(nums[10])'),
    ('Código correcto', 'def cuadrado(n):\n    return n ** 2\n\nprint(cuadrado(4))'),
]

for desc, code in test_cases:
    print(f"\n{'='*50}")
    print(f"TEST: {desc}")
    print(f"Código:\n{code}")
    print(f"\nFeedback:")
    print(review(code, pipe_ft))

## Comparación: base vs fine-tuned

Cargar el modelo base para comparar respuestas.

In [ ]:
# Descomentar para cargar base (requiere ~14GB VRAM adicional o descargar ft primero)
# pipe_base = pipeline(
#     'text-generation',
#     model=BASE_ID,
#     tokenizer=AutoTokenizer.from_pretrained(BASE_ID),
#     torch_dtype=torch.float16,
#     device_map='auto',
# )
#
# code_test = 'def suma(a, b):\n  return a - b'
# print('BASE:')
# print(review(code_test, pipe_base))
# print('\nFINE-TUNED:')
# print(review(code_test, pipe_ft))

## Demo interactiva con Gradio (en local)

In [ ]:
# !pip install -q gradio
# import gradio as gr
#
# demo = gr.Interface(
#     fn=lambda code, tokens: review(code, pipe_ft, int(tokens)),
#     inputs=[gr.Textbox(label='Código Python', lines=10), gr.Slider(64, 512, 256, step=64)],
#     outputs=gr.Textbox(label='Feedback', lines=8),
#     title='Code Reviewer en Español',
# )
# demo.launch(share=True)